# SatQuery: high-resolution models with the DINOv2 encoder (free Kaggle GPU)

Retrains the object finder and the question answerer for sharp aerial photos (VRSBench) on DINOv2-small features instead of ResNet-18. Same data, same test questions, so the scores compare directly with the app's current models.

**Before you start:** Settings → Accelerator **GPU T4** (not P100), Internet **On**. Then **Save Version → Save & Run All**.

1. Checks the GPU and the code, then downloads VRSBench (about 12 GB).
2. **Self-test** on a few real images. Look for `SELF-TEST PASSED` in the log.
3. The real run. Results are saved after every step.

Output tab: `satquery_hr_dino_models.zip`. **If a step fails, send `run_log.txt`**: it holds the real error.

In [ ]:
# Settings and the helper every step uses (nothing to change here)
import collections, json, os, subprocess, sys, time, urllib.parse, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

WORK = Path(os.environ.get("SQ_WORK", "/kaggle/working"))      # kept as the notebook's output
TMP = Path(os.environ.get("SQ_TMP", "/tmp/satquery"))          # downloads, thrown away afterwards
REPO = Path(os.environ.get("SQ_REPO", str(TMP / "SatQuery")))
ROOT = Path(os.environ.get("SQ_ROOT", str(TMP / "vrsbench")))   # VRSBench, about 12 GB
LIMIT = int(os.environ.get("SQ_LIMIT", "0"))                    # 0 = all images
WORK.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
LOG = WORK / "run_log.txt"
LOG.write_text("")
STATUS = {}

def say(text):
    print(text, flush=True)
    with LOG.open("a", encoding="utf-8") as fh:
        fh.write(text + "\n")

def pack():
    "Zip whatever exists so far, so a later failure never loses finished work."
    with zipfile.ZipFile(WORK / "satquery_hr_dino_models.zip", "w") as z:
        for f in ("dinov2_vits14.pt", "vrsbench_ground.pt", "vrsbench_ground.pt.metrics.json", "vrsbench_vqa.pt",
                  "vrsbench_vqa.pt.metrics.json", "run_log.txt"):
            if (WORK / f).exists():
                z.write(WORK / f, f)

def run(name, cmd, required=True, cwd=None):
    "Run a command, show its output live, keep it in run_log.txt, and show the real error if it fails."
    say(f"\n===== {name} =====")
    t0, tail = time.time(), collections.deque(maxlen=40)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        tail.append(line.rstrip())
        say(line.rstrip())
    ok = p.wait() == 0
    STATUS[name] = "ok" if ok else "FAILED"
    say(f"----- {name}: {STATUS[name]} in {(time.time() - t0) / 60:.1f} min")
    if not ok:
        last = tail[-1] if tail else "no output"
        say(f"\n##### {name} FAILED (exit code {p.returncode}). The real error is above; last line: {last}")
        if required:
            pack()
            raise RuntimeError(f"{name} failed: {last}  (full text in run_log.txt)")
    return ok

PY = sys.executable

In [ ]:
# Step 1a. Code: main branch, plus the encoder code if it is not on main yet
FIX = r'''diff --git a/satquery/config.py b/satquery/config.py
index e7ceb3e..4130160 100644
--- a/satquery/config.py
+++ b/satquery/config.py
@@ -44,6 +44,8 @@ class Settings:
 
     vrsbench_ground_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_VRSBENCH_GROUND_CKPT", "checkpoints/vrsbench_ground.pt"))
     vrsbench_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_VRSBENCH_CKPT", "checkpoints/vrsbench_vqa.pt"))
+    # DINOv2 weights, needed only by high-resolution checkpoints trained with that encoder
+    dino_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_DINO_CKPT", "checkpoints/dinov2_vits14.pt"))
     cdvqa_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_CDVQA_CKPT", "checkpoints/cdvqa.pt"))
     rsvqa_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_RSVQA_CKPT", "checkpoints/rsvqa_lr.pt"))
     flood_checkpoint: str = field(default_factory=lambda: os.getenv("SATQUERY_FLOOD_CKPT", "checkpoints/flood_seg.pt"))
diff --git a/satquery/models/dino.py b/satquery/models/dino.py
new file mode 100644
index 0000000..5e5a47e
--- /dev/null
+++ b/satquery/models/dino.py
@@ -0,0 +1,69 @@
+"""DINOv2 ViT-S/14 with registers (Oquab et al., 2023; Apache-2.0), frozen, as the image encoder of the dense
+high-resolution models. Parameter names follow the official release, so its weights load unchanged:
+https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_reg4_pretrain.pth
+A 392x392 image gives two 14x14x384 maps (block 9 and block 12, average-pooled from 28x28), stacked to 768 channels:
+the same shape as the ResNet-18 maps it replaces.
+"""
+from __future__ import annotations
+
+from pathlib import Path
+
+import torch
+import torch.nn as nn
+import torch.nn.functional as F
+
+DIM, DEPTH, HEADS, REGISTERS, PATCH, BASE = 384, 12, 6, 4, 14, 37
+INPUT = 392
+TAPS = (8, 11)      # blocks whose output becomes a feature map
+
+
+class _Block(nn.Module):
+    def __init__(self):
+        super().__init__()
+        self.norm1, self.norm2 = nn.LayerNorm(DIM, 1e-6), nn.LayerNorm(DIM, 1e-6)
+        self.attn = nn.ModuleDict({"qkv": nn.Linear(DIM, 3 * DIM), "proj": nn.Linear(DIM, DIM)})
+        self.mlp = nn.ModuleDict({"fc1": nn.Linear(DIM, 4 * DIM), "fc2": nn.Linear(4 * DIM, DIM)})
+        self.ls1 = nn.ParameterDict({"gamma": nn.Parameter(torch.ones(DIM))})
+        self.ls2 = nn.ParameterDict({"gamma": nn.Parameter(torch.ones(DIM))})
+
+    def forward(self, x: torch.Tensor) -> torch.Tensor:
+        b, n, d = x.shape
+        q, k, v = self.attn["qkv"](self.norm1(x)).reshape(b, n, 3, HEADS, d // HEADS).permute(2, 0, 3, 1, 4)
+        a = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(b, n, d)
+        x = x + self.ls1["gamma"] * self.attn["proj"](a)
+        return x + self.ls2["gamma"] * self.mlp["fc2"](F.gelu(self.mlp["fc1"](self.norm2(x))))
+
+
+class DinoV2S(nn.Module):
+    def __init__(self):
+        super().__init__()
+        self.cls_token = nn.Parameter(torch.zeros(1, 1, DIM))
+        self.register_tokens = nn.Parameter(torch.zeros(1, REGISTERS, DIM))
+        self.mask_token = nn.Parameter(torch.zeros(1, DIM))       # in the release; unused here
+        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + BASE * BASE, DIM))
+        self.patch_embed = nn.ModuleDict({"proj": nn.Conv2d(3, DIM, PATCH, PATCH)})
+        self.blocks = nn.ModuleList([_Block() for _ in range(DEPTH)])
+        self.norm = nn.LayerNorm(DIM, 1e-6)
+
+    def forward(self, x: torch.Tensor) -> torch.Tensor:
+        """(B, 3, INPUT, INPUT), ImageNet-normalised -> (B, 768, 14, 14)."""
+        x = self.patch_embed["proj"](x)
+        b, d, h, w = x.shape
+        pos = F.interpolate(self.pos_embed[:, 1:].reshape(1, BASE, BASE, d).permute(0, 3, 1, 2), size=(h, w),
+                            mode="bicubic", antialias=True)
+        x = torch.cat([(self.cls_token + self.pos_embed[:, :1]).expand(b, -1, -1),
+                       self.register_tokens.expand(b, -1, -1), (x + pos).flatten(2).transpose(1, 2)], 1)
+        maps = []
+        for i, blk in enumerate(self.blocks):
+            x = blk(x)
+            if i in TAPS:
+                m = self.norm(x)[:, 1 + REGISTERS:].transpose(1, 2).reshape(b, d, h, w)
+                maps.append(F.avg_pool2d(m, 2))
+        return torch.cat(maps, 1)
+
+
+def load_dino(path: str | Path) -> DinoV2S:
+    """The frozen encoder; half-precision weight files (ours) load back as float32."""
+    m = DinoV2S()
+    m.load_state_dict({k: v.float() for k, v in torch.load(path, map_location="cpu", weights_only=True).items()})
+    return m.eval().requires_grad_(False)
diff --git a/satquery/models/hr_ground_net.py b/satquery/models/hr_ground_net.py
index f26da7f..781ec8d 100644
--- a/satquery/models/hr_ground_net.py
+++ b/satquery/models/hr_ground_net.py
@@ -1,6 +1,7 @@
 """Dense referring-expression grounder for high-resolution aerial images (shared by training and the tool).
 
-Image: frozen ImageNet ResNet-18 at 448x448. layer3 (28x28x256, average-pooled to 14x14) and layer4
+Image: frozen encoder named in the checkpoint: DINOv2 ViT-S ("dinov2_s", satquery/models/dino.py) or, by default,
+ImageNet ResNet-18 at 448x448. layer3 (28x28x256, average-pooled to 14x14) and layer4
 (14x14x512) are stacked into a 14x14x768 map and projected to PROJ_DIM channels with a fixed PCA fitted on
 training images, so the cached maps fit in memory on a laptop.
 Expression: bag of unigrams/bigrams (`rsvqa_features.tokens`) through an EmbeddingBag.
@@ -14,12 +15,15 @@ from __future__ import annotations
 
 import functools
 import re
+from pathlib import Path
 
 import numpy as np
 import torch
 import torch.nn as nn
 import torch.nn.functional as F
 
+from satquery.config import settings
+from satquery.models import dino
 from satquery.models.hr_features import MEAN, STD
 from satquery.models.rsvqa_features import tokens
 
@@ -38,9 +42,29 @@ _V_SWAP = {"top": "bottom", "bottom": "top", "upper": "lower", "lower": "upper",
 
 
 # ---- image side --------------------------------------------------------------------------------------------
-@functools.lru_cache(maxsize=1)
-def load_dense_encoder() -> tuple[nn.Module, nn.Module]:
-    """Frozen ImageNet ResNet-18 split at layer3 / layer4; one shared copy per process (inference only)."""
+def dino_path() -> Path:
+    p = Path(settings.dino_checkpoint)
+    return p if p.is_absolute() else Path(__file__).resolve().parents[2] / p
+
+
+@functools.lru_cache(maxsize=8)
+def _encoder_name(path: str, mtime: float) -> str:
+    return torch.load(path, map_location="cpu", weights_only=False).get("encoder", "resnet18")
+
+
+def missing_encoder(checkpoint: Path) -> str | None:
+    """Why a dense checkpoint cannot run (its encoder's weights are not there), or None."""
+    if _encoder_name(str(checkpoint), checkpoint.stat().st_mtime) == "dinov2_s" and not dino_path().exists():
+        return f"encoder weights not found ({dino_path().name})"
+    return None
+
+
+@functools.lru_cache(maxsize=2)
+def load_dense_encoder(name: str = "resnet18"):
+    """The frozen encoder a checkpoint names; one shared copy per process (inference only). "resnet18" is a tuple:
+    ImageNet ResNet-18 split at layer3 / layer4."""
+    if name == "dinov2_s":
+        return dino.load_dino(dino_path())
     from torchvision import models
 
     m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval()
@@ -48,15 +72,19 @@ def load_dense_encoder() -> tuple[nn.Module, nn.Module]:
     return to_l3, m.layer4.eval()
 
 
-def encode_dense(enc: tuple[nn.Module, nn.Module], rgbs: list[np.ndarray]) -> np.ndarray:
+def encode_dense(enc, rgbs: list[np.ndarray]) -> np.ndarray:
     """HxWx3 uint8 arrays -> (N, RAW_DIM, GRID, GRID) float32 raw feature maps."""
     from PIL import Image
 
-    x = np.stack([np.asarray(Image.fromarray(a).convert("RGB").resize((INPUT, INPUT), Image.BILINEAR), np.float32)
+    vit = isinstance(enc, dino.DinoV2S)
+    side = dino.INPUT if vit else INPUT
+    x = np.stack([np.asarray(Image.fromarray(a).convert("RGB").resize((side, side), Image.BILINEAR), np.float32)
                   / 255.0 for a in rgbs])
-    dev = next(enc[0].parameters()).device          # CPU in the app; GPU when a trainer moved the encoder
+    dev = next((enc if vit else enc[0]).parameters()).device          # CPU in the app; GPU when a trainer moved the encoder
     x = torch.from_numpy((x - MEAN) / STD).permute(0, 3, 1, 2).to(dev)
     with torch.inference_mode():
+        if vit:
+            return enc(x).float().cpu().numpy()
         f3 = enc[0](x)
         f4 = enc[1](f3)
         return torch.cat([F.avg_pool2d(f3, 2), f4], 1).float().cpu().numpy()
diff --git a/satquery/tools/hrground.py b/satquery/tools/hrground.py
index 4b5ab3c..1be7008 100644
--- a/satquery/tools/hrground.py
+++ b/satquery/tools/hrground.py
@@ -51,7 +51,7 @@ def _load():
                 model = DenseGrounder(len(state["vocab"]))
                 model.load_state_dict(state["model"])
                 model.eval()
-                _cache[key] = (model, state, load_dense_encoder(), state.get("metrics", {}))
+                _cache[key] = (model, state, load_dense_encoder(state.get("encoder", "resnet18")), state.get("metrics", {}))
                 return _cache[key]
             nv, d = len(state["vocab"]), state["img_dim"]
 
@@ -109,6 +109,10 @@ class HighResGrounder(Tool):
     def available(self) -> tuple[bool, str]:
         if not _path(settings.vrsbench_ground_checkpoint).exists():
             return False, "checkpoint not found (vrsbench_ground.pt); run training/train_vrsbench_grounding.py"
+        from satquery.models.hr_ground_net import missing_encoder
+        why = missing_encoder(_path(settings.vrsbench_ground_checkpoint))
+        if why:
+            return False, why
         return True, "trained on VRSBench referring expressions"
 
     def run(self, ctx: ToolContext, params: dict[str, Any]) -> ToolResult:
diff --git a/satquery/tools/hrvqa.py b/satquery/tools/hrvqa.py
index ada652a..a713b9d 100644
--- a/satquery/tools/hrvqa.py
+++ b/satquery/tools/hrvqa.py
@@ -47,7 +47,7 @@ def _load():
             if state.get("arch") == "dense":      # 14x14 maps + attention (training/train_vrsbench_vqa_dense.py)
                 from satquery.models.hr_ground_net import DenseVQA, load_dense_encoder
                 model = DenseVQA(len(state["vocab"]), len(state["answers"]))
-                body = load_dense_encoder()
+                body = load_dense_encoder(state.get("encoder", "resnet18"))
             else:                                 # 3x3 grid (training/train_vrsbench_vqa.py)
                 model = build_model(len(state["vocab"]), img_dim=state["img_dim"], n_answers=len(state["answers"]),
                                     hidden=state.get("hidden", 512))
@@ -94,6 +94,10 @@ class HighResVQA(Tool):
     def available(self) -> tuple[bool, str]:
         if not _path(settings.vrsbench_checkpoint).exists():
             return False, "checkpoint not found (vrsbench_vqa.pt); run training/train_vrsbench_vqa.py"
+        from satquery.models.hr_ground_net import missing_encoder
+        why = missing_encoder(_path(settings.vrsbench_checkpoint))
+        if why:
+            return False, why
         return True, "trained on VRSBench"
 
     def run(self, ctx: ToolContext, params: dict[str, Any]) -> ToolResult:
diff --git a/training/train_vrsbench_grounding.py b/training/train_vrsbench_grounding.py
index 8710f60..19420f2 100644
--- a/training/train_vrsbench_grounding.py
+++ b/training/train_vrsbench_grounding.py
@@ -143,6 +143,9 @@ def main() -> None:
     ap.add_argument("--lr", type=float, default=1e-3)
     ap.add_argument("--min-count", type=int, default=3, help="keep expression tokens seen at least this often")
     ap.add_argument("--features-only", action="store_true")
+    ap.add_argument("--encoder", choices=("resnet18", "dinov2_s"), default="resnet18",
+                    help="frozen image encoder; use a separate --cache folder for each")
+    ap.add_argument("--limit", type=int, default=0, help="use only this many images per split (self-test)")
     ap.add_argument("--encode-bs", type=int, default=8,
                     help="images per encoder batch (memory peak grows with it; resumable at any value)")
     args = ap.parse_args()
@@ -156,12 +159,17 @@ def main() -> None:
     train, test = load_refs(args.root)
     tr_names = sorted({r["img"] for r in train})
     te_names = sorted({r["img"] for r in test})
+    if args.limit:
+        tr_names, te_names = tr_names[:args.limit], te_names[:args.limit]
+        train = [r for r in train if r["img"] in set(tr_names)]
+        test = [r for r in test if r["img"] in set(te_names)]
     print(f"{len(train)} train / {len(test)} test expressions over "
           f"{len(tr_names)} / {len(te_names)} images", flush=True)
 
     dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
     print("device:", dev, flush=True)
-    enc = tuple(m.to(dev) for m in load_dense_encoder())
+    enc = load_dense_encoder(args.encoder)
+    enc = tuple(m.to(dev) for m in enc) if args.encoder == "resnet18" else enc.to(dev)
     tr_imgs = ZipImages(args.root / "Images_train.zip")
     pca = fit_pca(cache / "dense_pca.npz", tr_imgs, tr_names, enc)
     F_tr = encode_all(cache / "train_dense.npy", tr_imgs, tr_names, enc, pca, args.encode_bs)
@@ -281,7 +289,7 @@ def main() -> None:
     table = []
     for chunk in np.array_split(order, 10):
         table.append({"p_max": round(float(probs[chunk].max()), 4), "acc": round(float(hit[chunk].mean()), 4)})
-    metrics = {"dataset": "VRSBench referring eval set", "model": "dense 14x14 FiLM grounder (hr_ground_net)",
+    metrics = {"dataset": "VRSBench referring eval set", "model": f"dense 14x14 FiLM grounder (hr_ground_net), {args.encoder} encoder",
                "test_acc_at_0.5": round(acc, 4), "test_mean_iou": round(miou, 4), "n_test": len(test),
                "held_out_acc_at_0.5": round(best, 4),
                "acc_by_class": {k: round(float(np.mean(v)), 3)
@@ -289,7 +297,7 @@ def main() -> None:
                "confidence_table": table, "pca_variance_kept": round(float(pca["explained"]), 4),
                "minutes": round((time.time() - t0) / 60, 1)}
     args.out.parent.mkdir(parents=True, exist_ok=True)
-    torch.save({"arch": "dense", "model": best_state, "vocab": vocab, "pca_mean": pca["mean"],
+    torch.save({"arch": "dense", "encoder": args.encoder, "model": best_state, "vocab": vocab, "pca_mean": pca["mean"],
                 "pca_comp": pca["comp"], "pca_scale": pca["scale"], "metrics": metrics}, args.out)
     Path(str(args.out) + ".metrics.json").write_text(json.dumps(metrics, indent=2))
     print(json.dumps(metrics, indent=2))
diff --git a/training/train_vrsbench_vqa_dense.py b/training/train_vrsbench_vqa_dense.py
index d633911..1bc6124 100644
--- a/training/train_vrsbench_vqa_dense.py
+++ b/training/train_vrsbench_vqa_dense.py
@@ -49,6 +49,8 @@ def main() -> None:
     ap.add_argument("--cache", type=Path, default=None, help="dense feature folder (default: <root>/features)")
     ap.add_argument("--out", type=Path, default=Path("checkpoints/vrsbench_vqa_dense.pt"))
     ap.add_argument("--answers", type=int, default=300)
+    ap.add_argument("--encoder", choices=("resnet18", "dinov2_s"), default="resnet18",
+                    help="the encoder that made the maps in --cache")
     ap.add_argument("--epochs", type=int, default=10)
     ap.add_argument("--bs", type=int, default=256)
     ap.add_argument("--lr", type=float, default=1e-3)
@@ -153,12 +155,12 @@ def main() -> None:
     model.load_state_dict(best_state)
     acc, per = evaluate("test")
     args.out.parent.mkdir(parents=True, exist_ok=True)
-    torch.save({"arch": "dense", "model": best_state, "vocab": vocab, "answers": answers,
+    torch.save({"arch": "dense", "encoder": args.encoder, "model": best_state, "vocab": vocab, "answers": answers,
                 "pca_mean": pca["mean"], "pca_comp": pca["comp"], "pca_scale": pca["scale"]}, args.out)
     metrics = {"dataset": "VRSBench VQA eval set (exact match after normalisation)", "test_accuracy": round(acc, 4),
                "test_per_type": {k: round(v, 4) for k, v in per.items()}, "n_test": len(test),
                "best_val_accuracy": round(best, 4), "answer_vocabulary": len(answers),
-               "model": "DenseVQA: 14x14 ResNet-18 maps (PCA 256) + FiLM + attention pooling",
+               "model": f"DenseVQA: 14x14 {args.encoder} maps (PCA 256) + FiLM + attention pooling",
                "minutes": round((time.time() - t0) / 60, 1)}
     Path(str(args.out) + ".metrics.json").write_text(json.dumps(metrics, indent=2))
     print(json.dumps(metrics, indent=2))
'''

if "SQ_REPO" not in os.environ:
    if not REPO.exists():
        run("get code", ["git", "clone", "-q", "--depth", "1", "-b", "main", "https://github.com/Vansh-Vaid/SatQuery.git", REPO])
    if (REPO / "satquery/models/dino.py").exists():
        say("encoder code: already on main")
    else:
        (TMP / "fix.patch").write_text(FIX, encoding="utf-8")
        run("add encoder code", ["git", "apply", "--whitespace=nowarn", TMP / "fix.patch"], cwd=REPO)

In [ ]:
# Step 1b. GPU check, and the DINOv2 weights (Apache-2.0, from Meta), stored in half precision for the app
CHECK = """
import sys, torch, torchvision
print("python", sys.version.split()[0], "| torch", torch.__version__)
if not torch.cuda.is_available():
    sys.exit("NO GPU: Settings -> Accelerator -> GPU T4, then Save & Run All again" if "--need-gpu" in sys.argv else 0)
try:
    torch.nn.Conv2d(3, 8, 3).cuda()(torch.rand(1, 3, 64, 64).cuda()).sum().backward()
except Exception as e:
    sys.exit(f"THIS GPU DOES NOT WORK WITH KAGGLE'S PYTORCH ({e}). Choose GPU T4 instead of P100.")
print("GPU ok:", torch.cuda.get_device_name(0))
"""
(TMP / "check.py").write_text(CHECK)
run("GPU check", [PY, TMP / "check.py"] + ([] if "SQ_REPO" in os.environ else ["--need-gpu"]))

DINO = WORK / "dinov2_vits14.pt"
os.environ["SATQUERY_DINO_CKPT"] = str(DINO)
WEIGHTS = """
import sys, time, urllib.request, torch
url = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_reg4_pretrain.pth"
for attempt in range(5):
    try:
        urllib.request.urlretrieve(url, sys.argv[1] + ".pth")
        break
    except Exception as e:
        print("attempt", attempt + 1, "failed:", e)
        time.sleep(10)
else:
    sys.exit("could not download the DINOv2 weights; is Settings -> Internet On?")
torch.save({k: v.half() for k, v in torch.load(sys.argv[1] + ".pth", map_location="cpu", weights_only=True).items()}, sys.argv[1])
from satquery.models.dino import load_dino
print("parameters", sum(p.numel() for p in load_dino(sys.argv[1]).parameters()))
"""
(TMP / "weights.py").write_text(WEIGHTS)
os.environ["PYTHONPATH"] = str(REPO)
if not DINO.exists():
    run("DINOv2 weights", [PY, TMP / "weights.py", DINO], cwd=REPO)
    Path(str(DINO) + ".pth").unlink(missing_ok=True)

In [ ]:
# Step 1c. VRSBench from Hugging Face (kept out of the notebook's output)
if "SQ_ROOT" not in os.environ:
    from huggingface_hub import hf_hub_download
    ROOT.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for f in ("VRSBench_train.json", "VRSBench_EVAL_vqa.json", "VRSBench_EVAL_referring.json", "Images_val.zip", "Images_train.zip"):
        for attempt in range(5):
            try:
                hf_hub_download("xiang709/VRSBench", f, repo_type="dataset", local_dir=str(ROOT))
                break
            except Exception as e:
                err = e
                say(f"{f}: attempt {attempt + 1} failed ({type(e).__name__}: {e}); retrying")
                time.sleep(15)
        else:
            pack()
            raise RuntimeError(f"could not download {f}: {err}")
        say(f"{f}: {(ROOT / f).stat().st_size / 1e9:.2f} GB ({(time.time() - t0) / 60:.1f} min)")

In [ ]:
# Step 2. SELF-TEST: both trainers on a few real images. It must pass before the long run starts.
def train(name, out, cache, limit=0, epochs=None, required=True):
    ep = ["--epochs", epochs] if epochs else []
    lim = ["--limit", limit] if limit else []
    ok = run(f"{name}: object finder", [PY, "training/train_vrsbench_grounding.py", "--root", ROOT, "--cache", cache, "--encoder", "dinov2_s",
                                        "--encode-bs", 64, "--out", out / "vrsbench_ground.pt", *lim, *ep], cwd=REPO, required=required)
    pack()
    ok = run(f"{name}: question answerer", [PY, "training/train_vrsbench_vqa_dense.py", "--root", ROOT, "--cache", cache, "--encoder", "dinov2_s",
                                            "--out", out / "vrsbench_vqa.pt", *ep], cwd=REPO, required=required) and ok
    pack()
    return ok

say("\n########## SELF-TEST (a few minutes) ##########")
train("self-test", TMP / "selftest", TMP / "selftest" / "features", limit=96, epochs=1)
say("\n########## SELF-TEST PASSED: every step works on real data (its scores mean nothing) ##########")

In [ ]:
# Step 3. The real run: encode every image once, then train both models
train("real run", WORK, TMP / "features", limit=LIMIT, epochs=1 if LIMIT else None)

In [ ]:
# Step 4. Verdict against the app's current models (same test sets: 16,159 expressions, 37,409 questions)
APP_GROUND, APP_VQA = 0.3849, 0.5599
APP_TYPES = {"image": 0.8893, "object category": 0.5409, "object color": 0.4932, "object direction": 0.413, "object existence": 0.8798,
             "object position": 0.3499, "object quantity": 0.4093, "object shape": 0.5559, "object size": 0.4955, "reasoning": 0.5333,
             "rural or urban": 0.8136, "scene type": 0.4814}
def read(f):
    return json.loads((WORK / f).read_text()) if (WORK / f).exists() else None

def verdict(new, app):
    return "BETTER" if new >= app + 0.02 else "not clearly better: keep the app's model"

g, v = read("vrsbench_ground.pt.metrics.json"), read("vrsbench_vqa.pt.metrics.json")
say("\n########## RESULTS ##########")
if g:
    say(f"object finder: {g['test_acc_at_0.5']:.4f} vs the app's {APP_GROUND:.4f} on {g['n_test']} expressions -> {verdict(g['test_acc_at_0.5'], APP_GROUND)}")
if v:
    say(f"questions: {v['test_accuracy']:.4f} vs the app's {APP_VQA:.4f} on {v['n_test']} questions -> {verdict(v['test_accuracy'], APP_VQA)}")
    for k, x in v["test_per_type"].items():
        say(f"   {k:18s} {x:.4f} vs {APP_TYPES.get(k, 0):.4f}{'   LOWER' if x < APP_TYPES.get(k, 0) - 0.01 else ''}")
if LIMIT:
    say("(LIMIT is set: these scores mean nothing)")
for name, state in STATUS.items():
    say(f"  {state:7s} {name}")
pack()
say("Done. Download satquery_hr_dino_models.zip from the Output tab.")

## Afterwards
Download `satquery_hr_dino_models.zip` and put it in the SatQuery folder. Nothing goes live from here: the models are first checked in the app for speed, memory and answers.